In [1]:
%pip install -q dotenv llama_stack_client==0.3.5 fire

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

from llama_stack_client import Agent
from llama_stack_client import AgentEventLogger
from llama_stack_client import LlamaStackClient

In [3]:
def stream_with_tools(response_stream):
    for chunk in response_stream:
        payload = getattr(getattr(chunk, "event", None), "payload", None)
        if not payload:
            continue
        step_type = getattr(payload, "step_type", "")
        if step_type == "tool_execution":
            print("\n🔧 Tool executed:")
            try:
                content = payload.delta.text
                print(content)
            except AttributeError:
                print(payload)
        elif getattr(payload, "event_type", "") == "step_progress":
            delta = getattr(getattr(payload, "delta", None), "text", None)
            if delta:
                print(delta, end="", flush=True)

In [4]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")
tavily_search_api_key = os.getenv("TAVILY_SEARCH_API_KEY")
wolfram_alpha_api_key = os.getenv("WOLFRAM_ALPHA_API_KEY")
model = "vllm/qwen3-8b"
client = LlamaStackClient(
    base_url=base_url,
    provider_data={"tavily_search_api_key": tavily_search_api_key, "wolfram_alpha_api_key": wolfram_alpha_api_key},
)

In [5]:
agent_config = {
    "model": "vllm/qwen3-8b",
    "instructions": """You are a helpful websearch assistant. When you are asked to search the latest you must use a tool. 
                     Whenever a tool is called, be sure return the response in a friendly and helpful tone.""",
    "enable_session_persistence": False,
    "toolgroups": ["builtin::websearch", "builtin::math"],
    "tool_choice": "auto",
    "sampling_params": {
        "max_tokens": 512,
    },
}


agent_response = client.alpha.agents.create(agent_config=agent_config)
agent_id = agent_response.agent_id

session_response = client.alpha.agents.session.create(
    agent_id=agent_id,
    session_name="websearch_test_session"
)
session_id = session_response.session_id

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1alpha/agents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1alpha/agents/d6ca5c93-2c38-42cf-9ab8-516eab52cf0b/session "HTTP/1.1 200 OK"


In [6]:
question = "Who is the current F1 World Champion?"

In [7]:
response = client.alpha.agents.turn.create(
    agent_id=agent_id,
    session_id=session_id,
    messages=[{"role": "user", "content": question}],
    stream=True,
)

stream_with_tools(response)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1alpha/agents/d6ca5c93-2c38-42cf-9ab8-516eab52cf0b/session/622ff9dd-e83a-4d10-8ebe-97dad6f060cf/turn "HTTP/1.1 200 OK"


<think>
Okay, the user is asking who the current F1 World Champion is. I need to find the most up-to-date information. Let me think about the tools available. There's brave_search for web searches and wolfram_alpha for computational knowledge. Since this is about the latest title holder, a web search would be more appropriate. I'll use brave_search with the query "current F1 World Champion". That should get the latest results. I don't think Wolfram Alpha is necessary here because it's more about real-time data rather than computational facts. Let me make sure the query is clear and concise. Alright, time to make the tool call.
</think>


🔧 Tool executed:
PayloadAgentTurnResponseStepStartPayload(event_type='step_start', step_id='c9c7d042-b0a5-4761-8ebc-75096660a6cb', step_type='tool_execution', metadata={})

🔧 Tool executed:
PayloadAgentTurnResponseStepProgressPayload(delta=PayloadAgentTurnResponseStepProgressPayloadDeltaToolCallDelta(parse_status='in_progress', tool_call=ToolCall(argum

In [8]:
question = "(7 * 12 ^ 10) / 321 ? and How many calories are there in a pound of strawberries?"

In [9]:
response = client.alpha.agents.turn.create(
    agent_id=agent_id,
    session_id=session_id,
    messages=[{"role": "user", "content": question}],
    stream=True,
)

stream_with_tools(response)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1alpha/agents/d6ca5c93-2c38-42cf-9ab8-516eab52cf0b/session/622ff9dd-e83a-4d10-8ebe-97dad6f060cf/turn "HTTP/1.1 200 OK"


<think>
Okay, the user asked two separate questions. Let me tackle them one by one.

First, the math problem: (7 * 12^10) / 321. Hmm, that's a big exponent. I need to calculate 12^10 first. Let me recall that 12^10 is 12 multiplied by itself ten times. But doing that manually would take time. Maybe I can use Wolfram Alpha for accurate computation. The user might need the exact value, so using the tool is the best bet.

Second question: How many calories are in a pound of strawberries? I know strawberries are low in calories, but I need the exact number. I should check a reliable source. Maybe a nutrition database or a calculator. Since the user mentioned "a pound," I should convert pounds to grams if necessary, but maybe the tool can handle that. Again, using Wolfram Alpha could give precise data. Alternatively, a quick search might provide the average calorie count per pound. Let me verify both questions with the appropriate tools.
</think>



🔧 Tool executed:
PayloadAgentTurnResponse